# Practical Application III: Comparing Classifiers

**Overview**: In this practical application, your goal is to compare the performance of the classifiers we encountered in this section, namely K Nearest Neighbor, Logistic Regression, Decision Trees, and Support Vector Machines.  We will utilize a dataset related to marketing bank products over the telephone.  



### Getting Started

Our dataset comes from the UCI Machine Learning repository [link](https://archive.ics.uci.edu/ml/datasets/bank+marketing).  The data is from a Portugese banking institution and is a collection of the results of multiple marketing campaigns.  We will make use of the article accompanying the dataset [here](CRISP-DM-BANK.pdf) for more information on the data and features.



### Problem 1: Understanding the Data

To gain a better understanding of the data, please read the information provided in the UCI link above, and examine the **Materials and Methods** section of the paper.  How many marketing campaigns does this data represent?

The dataset represents **17 marketing campaigns** conducted by the Portuguese banking institution between May 2008 and November 2010.


### Problem 2: Read in the Data

Use pandas to read in the dataset `bank-additional-full.csv` and assign to a meaningful variable name.

In [ ]:
import time

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42


In [ ]:
df = pd.read_csv('data/bank-additional-full.csv', sep=';')


In [ ]:
df.head()

### Problem 3: Understanding the Features


Examine the data description below, and determine if any of the features are missing values or need to be coerced to a different data type.


```
Input variables:
# bank client data:
1 - age (numeric)
2 - job : type of job (categorical: 'admin.','blue-collar','entrepreneur','housemaid','management','retired','self-employed','services','student','technician','unemployed','unknown')
3 - marital : marital status (categorical: 'divorced','married','single','unknown'; note: 'divorced' means divorced or widowed)
4 - education (categorical: 'basic.4y','basic.6y','basic.9y','high.school','illiterate','professional.course','university.degree','unknown')
5 - default: has credit in default? (categorical: 'no','yes','unknown')
6 - housing: has housing loan? (categorical: 'no','yes','unknown')
7 - loan: has personal loan? (categorical: 'no','yes','unknown')
# related with the last contact of the current campaign:
8 - contact: contact communication type (categorical: 'cellular','telephone')
9 - month: last contact month of year (categorical: 'jan', 'feb', 'mar', ..., 'nov', 'dec')
10 - day_of_week: last contact day of the week (categorical: 'mon','tue','wed','thu','fri')
11 - duration: last contact duration, in seconds (numeric). Important note: this attribute highly affects the output target (e.g., if duration=0 then y='no'). Yet, the duration is not known before a call is performed. Also, after the end of the call y is obviously known. Thus, this input should only be included for benchmark purposes and should be discarded if the intention is to have a realistic predictive model.
# other attributes:
12 - campaign: number of contacts performed during this campaign and for this client (numeric, includes last contact)
13 - pdays: number of days that passed by after the client was last contacted from a previous campaign (numeric; 999 means client was not previously contacted)
14 - previous: number of contacts performed before this campaign and for this client (numeric)
15 - poutcome: outcome of the previous marketing campaign (categorical: 'failure','nonexistent','success')
# social and economic context attributes
16 - emp.var.rate: employment variation rate - quarterly indicator (numeric)
17 - cons.price.idx: consumer price index - monthly indicator (numeric)
18 - cons.conf.idx: consumer confidence index - monthly indicator (numeric)
19 - euribor3m: euribor 3 month rate - daily indicator (numeric)
20 - nr.employed: number of employees - quarterly indicator (numeric)

Output variable (desired target):
21 - y - has the client subscribed a term deposit? (binary: 'yes','no')
```



In [ ]:
print(df.shape)
display(df.info())
display(df.isna().sum())
display(df.nunique().sort_values())


### Problem 4: Understanding the Task

After examining the description and data, your goal now is to clearly state the *Business Objective* of the task.  State the objective below.

In [ ]:
df.info()

In [ ]:
unknown_counts = (df == 'unknown').sum().sort_values(ascending=False)
unknown_counts[unknown_counts > 0]


There are no `NaN` values in the file, but some categorical columns use the string value `unknown` to represent missing or unavailable information. These should be treated as categorical levels or handled with imputation/removal. The numeric columns are already stored as numeric types, and the categorical columns should be encoded before modeling.

The business objective is to predict whether a contacted client will subscribe to a term deposit (`y = yes`). A useful model would help the bank prioritize likely subscribers, reduce unnecessary calls, and improve campaign efficiency.


### Problem 5: Engineering Features

Now that you understand your business objective, we will build a basic model to get started.  Before we can do this, we must work to encode the data.  Using just the bank information features, prepare the features and target column for modeling with appropriate encoding and transformations.

In [ ]:
# Start with only the bank client information features requested in the prompt.
bank_features = ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan']
target = 'y'

X = df[bank_features]
y = df[target].map({'no': 0, 'yes': 1})


In [ ]:
numeric_features = ['age']
categorical_features = ['job', 'marital', 'education', 'default', 'housing', 'loan']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ]
)


In [ ]:
X.head()


### Problem 6: Train/Test Split

With your data prepared, split it into a train and test set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)


In [ ]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape


In [ ]:
y_train.value_counts(normalize=True), y_test.value_counts(normalize=True)


### Problem 7: A Baseline Model

Before we build our first model, we want to establish a baseline.  What is the baseline performance that our classifier should aim to beat?

In [ ]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
baseline_accuracy = baseline.score(X_test, y_test)
baseline_accuracy


In [ ]:
y_test.value_counts(normalize=True)


The baseline classifier predicts the majority class (`no`) for every observation. Because about 88.7% of clients did not subscribe, a model needs to beat approximately **88.7% accuracy** to improve on this simple baseline. Accuracy alone is limited here because the classes are imbalanced.


### Problem 8: A Simple Model

Use Logistic Regression to build a basic model on your data.  

In [ ]:
logreg_pipe = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]
)

logreg_pipe.fit(X_train, y_train)


### Problem 9: Score the Model

What is the accuracy of your model?

In [ ]:
logreg_train_accuracy = logreg_pipe.score(X_train, y_train)
logreg_test_accuracy = logreg_pipe.score(X_test, y_test)

print(f'Train accuracy: {logreg_train_accuracy:.4f}')
print(f'Test accuracy:  {logreg_test_accuracy:.4f}')


### Problem 10: Model Comparisons

Now, we aim to compare the performance of the Logistic Regression model to our KNN algorithm, Decision Tree, and SVM models.  Using the default settings for each of the models, fit and score each.  Also, be sure to compare the fit time of each of the models.  Present your findings in a `DataFrame` similar to that below:

| Model | Train Time | Train Accuracy | Test Accuracy |
| ----- | ---------- | -------------  | -----------   |
|     |    |.     |.     |

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'KNN': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'SVM': SVC(random_state=RANDOM_STATE),
}


In [ ]:
model_results = []

for model_name, model in models.items():
    pipe = Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('model', model),
        ]
    )
    
    start = time.perf_counter()
    pipe.fit(X_train, y_train)
    train_time = time.perf_counter() - start
    
    model_results.append(
        {
            'Model': model_name,
            'Train Time': train_time,
            'Train Accuracy': pipe.score(X_train, y_train),
            'Test Accuracy': pipe.score(X_test, y_test),
        }
    )

results_df = pd.DataFrame(model_results).sort_values('Test Accuracy', ascending=False)
results_df


Using only the bank client features, the default models are not very strong. Logistic Regression and SVM are close to the majority-class baseline, while KNN and the Decision Tree perform worse on the test set. The Decision Tree also shows signs of overfitting because its train accuracy is higher than its test accuracy.


### Problem 11: Improving the Model

Now that we have some basic models on the board, we want to try to improve these.  Below, we list a few things to explore in this pursuit.


- Hyperparameter tuning and grid search.  All of our models have additional hyperparameters to tune and explore.  For example the number of neighbors in KNN or the maximum depth of a Decision Tree.  
- Adjust your performance metric

For model improvement, I use a more realistic feature set: all available columns **except** `duration`, because call duration is not known before the call and would leak information from the completed contact. I also evaluate balanced accuracy and ROC AUC because the target classes are imbalanced.


In [ ]:
realistic_features = [col for col in df.columns if col not in ['duration', 'y']]
X_realistic = df[realistic_features]
y_realistic = df['y'].map({'no': 0, 'yes': 1})

numeric_realistic = X_realistic.select_dtypes(include='number').columns.tolist()
categorical_realistic = X_realistic.select_dtypes(exclude='number').columns.tolist()

realistic_preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_realistic),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_realistic),
    ]
)

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_realistic,
    y_realistic,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y_realistic,
)


In [ ]:
search_spaces = {
    'Logistic Regression': (
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, class_weight='balanced'),
        {'model__C': [0.01, 0.1, 1, 10]},
    ),
    'Decision Tree': (
        DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
        {'model__max_depth': [3, 5, 8, 12], 'model__min_samples_leaf': [50, 100, 250]},
    ),
    'KNN': (
        KNeighborsClassifier(),
        {'model__n_neighbors': [5, 15, 31], 'model__weights': ['uniform', 'distance']},
    ),
}


In [ ]:
tuned_results = []
best_search = None
best_balanced_accuracy = -np.inf

for model_name, (model, param_grid) in search_spaces.items():
    pipe = Pipeline(
        steps=[
            ('preprocessor', realistic_preprocessor),
            ('model', model),
        ]
    )
    
    search = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring='balanced_accuracy',
        cv=3,
        n_jobs=1,
    )
    
    start = time.perf_counter()
    search.fit(X_train_r, y_train_r)
    search_time = time.perf_counter() - start
    
    y_pred = search.predict(X_test_r)
    y_proba = search.predict_proba(X_test_r)[:, 1]
    balanced_acc = balanced_accuracy_score(y_test_r, y_pred)
    
    tuned_results.append(
        {
            'Model': model_name,
            'Best Params': search.best_params_,
            'Search Time': search_time,
            'Test Accuracy': accuracy_score(y_test_r, y_pred),
            'Balanced Accuracy': balanced_acc,
            'ROC AUC': roc_auc_score(y_test_r, y_proba),
        }
    )
    
    if balanced_acc > best_balanced_accuracy:
        best_balanced_accuracy = balanced_acc
        best_search = search

tuned_results_df = pd.DataFrame(tuned_results).sort_values('Balanced Accuracy', ascending=False)
tuned_results_df


In [ ]:
best_model = best_search.best_estimator_
y_pred_best = best_model.predict(X_test_r)
y_proba_best = best_model.predict_proba(X_test_r)[:, 1]

print('Best model:', tuned_results_df.iloc[0]['Model'])
print('Best parameters:', best_search.best_params_)
print(f"Accuracy: {accuracy_score(y_test_r, y_pred_best):.4f}")
print(f"Balanced accuracy: {balanced_accuracy_score(y_test_r, y_pred_best):.4f}")
print(f"ROC AUC: {roc_auc_score(y_test_r, y_proba_best):.4f}")


In [ ]:
print(classification_report(y_test_r, y_pred_best, target_names=['no', 'yes']))


In [ ]:
confusion_matrix_df = pd.DataFrame(
    confusion_matrix(y_test_r, y_pred_best),
    index=['Actual no', 'Actual yes'],
    columns=['Predicted no', 'Predicted yes'],
)
confusion_matrix_df


The tuned Logistic Regression model is the best option from this search by balanced accuracy. Its overall accuracy is lower than the majority-class baseline, but it identifies many more actual subscribers. That tradeoff is reasonable for direct marketing because the business value comes from finding likely `yes` customers, not merely predicting `no` for almost everyone.


SVM was included in the default model comparison, but I did not include it in the grid search because the default SVM was already much slower than the other models on the full dataset. A larger SVM search would be possible on the smaller `bank-additional.csv` sample or with a carefully constrained parameter grid.


In [ ]:
# Optional: inspect coefficients from the tuned Logistic Regression model.
feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()
coefficients = best_model.named_steps['model'].coef_[0]

coef_df = pd.DataFrame(
    {
        'feature': feature_names,
        'coefficient': coefficients,
        'absolute_coefficient': np.abs(coefficients),
    }
).sort_values('absolute_coefficient', ascending=False)

coef_df.head(15)


The most useful next step would be to align the scoring metric with campaign economics. For example, if calling a false positive is inexpensive compared with missing a likely subscriber, recall or lift for the `yes` class should receive more emphasis than raw accuracy.


##### Questions

The CRISP-DM process helped frame this as a business problem first, rather than just a modeling exercise. The baseline model shows that raw accuracy can be deceptive because most clients say `no`. With only bank-client demographic/account features, none of the default classifiers substantially improves on the baseline. After adding realistic campaign, contact-history, and economic context features while excluding `duration`, tuned Logistic Regression provides the strongest balanced result in this notebook.
